# Playing with the stochastic reservoir

This notebook takes the reservoir of [Stochastic Dynamic Programming](https://pierrelucbacon.com/rlbook/stochastic-dp/) and lets you break it in four ways, each tied to a point from the lecture of October 26:

1. only the *ordering* of the cost-to-go matters for the policy;
2. Monte Carlo can replace the exact expectation in backward induction;
3. the expectation is a choice that makes us risk neutral, and CVaR is one alternative;
4. with a finite horizon, the optimal policy depends on time.

Every experiment starts with a **Predict first** question. Write your guess down before you run the cell. The whole notebook runs in under a minute on a laptop CPU, and it needs only NumPy, Matplotlib and ipywidgets.

## The model

This is the model of the chapter, unchanged: 52 weekly stages, storage $x$ between 200 and 2000 hm³, a release $u$ chosen before the week's inflow is known, and an inflow $w_t = \xi_t \bar w_t$ whose factor $\xi_t$ is 0.6, 1.0 or 1.4 with probabilities 1/4, 1/2, 1/4. The week earns $p_t\,\kappa\,h(x)\,\tilde u$, where $\tilde u$ is the release that actually happens.

The solver has one change from the chapter. In the chapter, the bracket of Bellman's equation is averaged over the three inflows. Here, the function that combines the three outcomes is an argument, `aggregate`, and the expectation is only its default. A second argument, `transform`, lets you tamper with $J_{t+1}$ before it enters the recursion.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

T = 52
weeks = np.arange(T)
w_bar = 60 + 340 * np.exp(-((weeks - 19) / 3.5) ** 2) + 60 * np.exp(-((weeks - 43) / 4) ** 2)
w_bar = 10 * np.round(w_bar / 10)                    # inflow forecast (hm³/week)
p = 60 + 30 * np.cos(2 * np.pi * (weeks - 5) / 52)   # price ($/MWh)
x_min, x_max = 200, 2000
u_min, u_max = 30, 250
kappa, p_T = 2.45, 50

def head(x):
    return 40 + 40 * np.sqrt(x / x_max)

def terminal_value(x):
    return p_T * kappa * head(x) * (x - x_min)

dx = 10
X = np.arange(x_min, x_max + dx, dx)
U = np.arange(u_min, u_max + dx, dx)
i0 = np.searchsorted(X, 1100)                        # initial storage: 1100 hm³

factors = np.array([0.6, 1.0, 1.4])
probs = np.array([0.25, 0.5, 0.25])

def expectation(outcomes, probs):
    """outcomes[k] is the bracket of Bellman's equation under inflow factor k."""
    return np.tensordot(probs, outcomes, axes=1)

def solve(factors=factors, probs=probs, aggregate=expectation, transform=None):
    J = np.zeros((T + 1, len(X)))
    policy = np.zeros((T, len(X)))
    J[T] = terminal_value(X)
    x, u = X[:, None], U[None, :]
    for t in reversed(range(T)):
        J_next = J[t + 1] if transform is None else transform(t, J[t + 1])
        outcomes = []
        for factor in factors:
            inflow = factor * w_bar[t]
            release = np.minimum(u, x + inflow - x_min)       # turbines stop at dead storage
            x_next = np.minimum(x + inflow - release, x_max)  # water above capacity spills
            outcomes.append(p[t] * kappa * head(x) * release + np.interp(x_next, X, J_next))
        value = aggregate(np.array(outcomes), np.asarray(probs))
        J[t] = value.max(axis=1)
        policy[t] = U[value.argmax(axis=1)]
    return J, policy

def simulate(policy, years, x0=1100):
    """Revenue of each simulated year (one row of inflow factors per year), in M$."""
    x = np.full(len(years), float(x0))
    revenue = 0.0
    for t in range(T):
        inflow = years[:, t] * w_bar[t]
        requested = policy[t, np.round((x - x_min) / dx).astype(int)]
        release = np.minimum(requested, x + inflow - x_min)
        revenue = revenue + p[t] * kappa * head(x) * release
        x = np.minimum(x + inflow - release, x_max)
    return (revenue + terminal_value(x)) / 1e6

def worst(revenue, fraction=0.05):
    """Average revenue of the worst 5% of years."""
    return np.sort(revenue)[: int(fraction * len(revenue))].mean()

rng = np.random.default_rng(0)
years = rng.choice(factors, size=(2000, T), p=probs)   # every policy below faces these same years

J_star, policy_star = solve()
revenue_star = simulate(policy_star, years)
print(f"predicted J_0(1100) = {J_star[0, i0] / 1e6:.2f} M$")
print(f"simulated mean      = {revenue_star.mean():.2f} M$, worst 5% of years = {worst(revenue_star):.2f} M$")

## 1. Only the ordering matters

In class: the policy is read off an argmax, and adding a constant to $J_{t+1}$ does not change which release wins. So the cost-to-go only has to *rank* releases correctly. Let us check that, then see how far we can push it.

In [ ]:
_, policy_shifted = solve(transform=lambda t, J: J + 1e6)
print("same policy after adding 1 M$ to every J_{t+1}:", np.array_equal(policy_shifted, policy_star))

**Predict first:** if you multiply $J_{t+1}$ by 0.9 or 1.1 instead, a uniform 10% error, will the policy change much? Will the revenue?

In [ ]:
for scale in [0.9, 0.99, 1.0, 1.01, 1.1]:
    _, policy = solve(transform=lambda t, J, s=scale: s * J)
    changed = np.mean(policy != policy_star)
    print(f"scale {scale:4}: {changed:5.1%} of decisions change, mean revenue {simulate(policy, years).mean():6.2f} M$")

**Predict first:** now add independent noise to $J_{t+1}$ at each grid point, at every stage. $J$ is in the tens of millions of dollars. How large can the noise be before the reservoir loses 1% of its revenue?

In [ ]:
for sigma in [1e3, 1e4, 3e4, 1e5, 3e5]:
    noise_rng = np.random.default_rng(1)
    _, policy = solve(transform=lambda t, J, s=sigma: J + s * noise_rng.standard_normal(len(J)))
    changed = np.mean(policy != policy_star)
    print(f"noise {sigma:8.0f} $: {changed:5.1%} of decisions change, mean revenue {simulate(policy, years).mean():6.2f} M$")

**Think about it.**

- Why does a *uniform* 10% error do so much damage, when random noise that changes a quarter of the decisions costs almost nothing? Look at what the scaling does to the trade-off between the revenue this week and the value of the water left.
- Which transformations of $J_{t+1}$ leave the policy exactly optimal? Is "any increasing function of $J$" one of them?
- When we fit $J$ with a neural network later in the course, the loss will measure how close $J$ is to its target everywhere. Is that the right thing to measure?

## 2. Monte Carlo instead of the exact expectation

Here the expectation is a sum of three terms, so we can compute it exactly. In most problems it is an integral we cannot compute. The fix from class is Monte Carlo: at each stage, draw $N$ inflow factors and average over them. The function below is the solver of the chapter with the three-term sum replaced by a sample average.

**Predict first:** as $N$ grows, which one settles down first, the predicted value $J_0(1100)$ or the revenue of the policy?

In [ ]:
def solve_monte_carlo(N, seed):
    draws = np.random.default_rng(seed).choice(factors, size=(T, N), p=probs)  # fresh draws at each stage
    J = np.zeros((T + 1, len(X)))
    policy = np.zeros((T, len(X)))
    J[T] = terminal_value(X)
    x, u = X[:, None], U[None, :]
    for t in reversed(range(T)):
        value = np.zeros((len(X), len(U)))
        for factor in draws[t]:                         # a sample average replaces the expectation
            inflow = factor * w_bar[t]
            release = np.minimum(u, x + inflow - x_min)
            x_next = np.minimum(x + inflow - release, x_max)
            value += (p[t] * kappa * head(x) * release + np.interp(x_next, X, J[t + 1])) / N
        J[t] = value.max(axis=1)
        policy[t] = U[value.argmax(axis=1)]
    return J, policy

print(f"exact: predicted {J_star[0, i0] / 1e6:.2f} M$, simulated {revenue_star.mean():.2f} M$\n")
print("   N   predicted J_0 (mean ± sd over 10 seeds)   simulated revenue (mean ± sd)")
for N in [1, 3, 10, 30, 100, 300]:
    runs = [solve_monte_carlo(N, seed) for seed in range(10)]
    predicted = np.array([J[0, i0] / 1e6 for J, _ in runs])
    simulated = np.array([simulate(policy, years).mean() for _, policy in runs])
    print(f"{N:4d}   {predicted.mean():6.2f} ± {predicted.std():4.2f}                       "
          f"{simulated.mean():6.2f} ± {simulated.std():4.2f}")

**Think about it.**

- The revenue of the policy stops improving long before the predicted value stops moving. Connect this to experiment 1.
- Suppose the inflow factor were a vector of 1000 tributaries instead of one number. What happens to the cost of the exact sum? What happens to the cost of the Monte Carlo average for a fixed $N$?
- `solve_monte_carlo` never looks at `probs` except to draw samples. What does that let you replace the inflow model with?

## 3. Risk neutral or risk averse?

The expectation ranks policies by their mean and ignores everything else about the distribution of outcomes. One alternative from class is the conditional value at risk: $\mathrm{CVaR}_\alpha$ is the average of the worst $\alpha$ fraction of outcomes. With $\alpha = 1$ it is the expectation. With $\alpha = 0.25$ and our three inflows, it is the outcome of the dry week alone.

The aggregator below replaces the expectation by $\mathrm{CVaR}_\alpha$ at every stage. This is called a *nested* risk measure: each week the operator plans as if the next week's inflow will be bad.

In [ ]:
def cvar(alpha):
    def aggregate(outcomes, probs):
        order = np.argsort(outcomes, axis=0)               # worst revenue first
        sorted_outcomes = np.take_along_axis(outcomes, order, axis=0)
        q = probs[order]
        before = np.cumsum(q, axis=0) - q                  # probability of the outcomes ranked before
        weight = np.clip(np.minimum(q, alpha - before), 0, None)
        return (weight * sorted_outcomes).sum(axis=0) / alpha
    return aggregate

from ipywidgets import interact, FloatSlider

@interact(alpha=FloatSlider(value=0.25, min=0.05, max=1.0, step=0.05))
def compare(alpha):
    _, policy = solve(aggregate=cvar(alpha))
    revenue = simulate(policy, years)
    print(f"risk neutral : mean {revenue_star.mean():.2f} M$, worst 5% of years {worst(revenue_star):.2f} M$")
    print(f"CVaR_{alpha:.2f}   : mean {revenue.mean():.2f} M$, worst 5% of years {worst(revenue):.2f} M$")
    bins = np.linspace(min(revenue.min(), revenue_star.min()), max(revenue.max(), revenue_star.max()), 50)
    plt.figure(figsize=(7, 3))
    plt.hist(revenue_star, bins, alpha=0.6, label="risk neutral")
    plt.hist(revenue, bins, alpha=0.6, label=f"nested CVaR, alpha={alpha:.2f}")
    plt.xlabel("yearly revenue (M$)")
    plt.legend()
    plt.show()

**Predict first:** the risk-averse operator gives up some mean revenue. Does it at least do better in the worst 5% of years?

**Think about it.**

- Run it. Were you right? Plot the yearly revenue against the total inflow of that year, `(years * w_bar).sum(axis=1)`. What decides whether a year is bad, and can any release rule change that?
- Nested CVaR is pessimistic about *every week*. CVaR of the *yearly revenue* is pessimistic about the year as a whole. Why can't you get the second one by plugging a different aggregator into backward induction? (Hint: what would the operator need to remember about the weeks already played?)
- Change the model so that risk aversion does pay off. For example, make droughts rarer but much worse, or add a penalty when storage falls below some level. What did you change, and why does it help?

## 4. The policy depends on time

With a finite horizon, the optimal release depends on the week as well as on the storage. Below is the optimal release at the same storage, week by week.

**Predict first:** at 1100 hm³, should the release in the last week of December be larger or smaller than in late October?

In [ ]:
storages = [600, 1100, 1600]
plt.figure(figsize=(7, 3))
for s in storages:
    plt.step(weeks, policy_star[:, np.searchsorted(X, s)], where="post", label=f"x = {s} hm³")
plt.xlabel("week")
plt.ylabel("optimal release (hm³)")
plt.legend()
plt.show()

**Think about it.**

- What in the model makes the same storage worth releasing in one week and keeping in another? Point to the line of code.
- Set `p_T = 0` (the water left at the end is worthless) and rerun. What happens in the last weeks? Is that a property of the reservoir or of the horizon?
- Next week we move to the infinite horizon. What would have to be true of $p_t$ and $\bar w_t$ for a policy without $t$ to be optimal?

In [ ]:
# Checks: these should still pass after you change the experiments above.
_, policy_check = solve(transform=lambda t, J: J - 5e5)
assert np.array_equal(policy_check, policy_star)
assert np.allclose(cvar(1.0)(np.array([[1.0], [2.0], [4.0]]), probs), 0.25 * 1 + 0.5 * 2 + 0.25 * 4)
assert np.isclose(cvar(0.25)(np.array([[1.0], [2.0], [4.0]]), probs), 1.0)
print("Checks passed.")